# 📚 Define Libraries

In [1]:
from google.cloud import bigquery
from datetime import datetime

# ---------------------------------------------------------
# 1. Create BigQuery client
# ---------------------------------------------------------
client = bigquery.Client(project="prj-prod-dataplatform")

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


# ❄️ beta_demo_model_cash_v1

In [2]:
print("Starting BigQuery job...")

sq = """ 
create or replace table `dap_ds_poweruser_playground.beta_demo_model_cash_v1` as 
WITH
  parsed AS (
    (
      SELECT
        safe_cast(customerId as string) customerId ,
        digitalLoanAccountId,
        modelDisplayName,
        modelVersionId,
        safe_cast(prediction as numeric) prediction,
        trenchCategory,
        REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeatures,
        'Prod' Data_selection,
        deviceOs osType,
      FROM `prj-prod-dataplatform.audit_balance.ml_model_run_details`
      WHERE
        modelDisplayName IN ('Beta-Cash-Demo-Model', 'beta_demo_model_cash')
        AND modelVersionId = 'v1'
    )
    UNION ALL
    (
      SELECT
        safe_cast(customerId as string) customerId,
        digitalLoanAccountId,
        modelDisplayName,
        modelVersionId,
        safe_cast(prediction as numeric) prediction,
        trenchCategory,
        REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeatures,
        Data_selection,
        deviceOs osType,
      FROM
        prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
      WHERE
        modelDisplayName IN ('Beta-Cash-Demo-Model', 'beta_demo_model_cash')
        AND modelVersionId = 'v1'
    )
  )
SELECT
    t.* EXCEPT(calcFeatures),

    SAFE_CAST(JSON_VALUE(t.calcFeatures, '$.ln_vas_opted_flag') AS INT64)
        AS ln_vas_opted_flag,

    SAFE_CAST(JSON_VALUE(t.calcFeatures, '$.ln_self_dec_income') AS INT64)
        AS ln_self_dec_income,

    SAFE_CAST(JSON_VALUE(t.calcFeatures, '$.ln_age') AS INT64)
        AS ln_age,

    JSON_VALUE(t.calcFeatures, '$.ln_source_funds_new_bin')
        AS ln_source_funds_new_bin,

    JSON_VALUE(t.calcFeatures, '$.ln_loan_level_user_type')
        AS ln_loan_level_user_type,

    JSON_VALUE(t.calcFeatures, '$.ln_industry_new_cat_bin')
        AS ln_industry_new_cat_bin,

    JSON_VALUE(t.calcFeatures, '$.ln_marital_status')
        AS ln_marital_status,

    JSON_VALUE(t.calcFeatures, '$.ln_doc_type_rolled')
        AS ln_doc_type_rolled,

    JSON_VALUE(t.calcFeatures, '$.ln_education_level')
        AS ln_education_level,

    JSON_VALUE(t.calcFeatures, '$.ln_ref2_type')
        AS ln_ref2_type,

    JSON_VALUE(t.calcFeatures, '$.ln_email_primary_domain')
        AS ln_email_primary_domain,

    JSON_VALUE(t.calcFeatures, '$.ln_os_type')
        AS ln_os_type,

    JSON_VALUE(t.calcFeatures, '$.ln_mature_fspd30_flag')
        AS ln_mature_fspd30_flag,

    JSON_VALUE(t.calcFeatures, '$.ln_fspd30_flag')
        AS ln_fspd30_flag,

    SAFE_CAST(
        JSON_VALUE(t.calcFeatures, '$.ln_appln_submit_datetime')
        AS TIMESTAMP
    ) AS ln_appln_submit_datetime,

    JSON_VALUE(t.calcFeatures, '$.ln_province_bin')
        AS ln_province_bin,

    SAFE_CAST(
        JSON_VALUE(t.calcFeatures, '$.Beta_Cash_Demo_Score')
        AS FLOAT64
    ) AS Beta_Cash_Demo_Score,
    date(lmt.startApplyDateTime) application_date
FROM parsed t
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = t.digitalLoanAccountId
;

"""

query_job = client.query(sq)
# Wait until query finishes
query_job.result()
print("BigQuery job completed successfully!")


# ---------------------------------------------------------
# 3. Job details
# ---------------------------------------------------------
print("\n========== JOB DETAILS ==========")

print(f"Job ID              : {query_job.job_id}")
print(f"State               : {query_job.state}")
print(f"Created             : {query_job.created}")
print(f"Started             : {query_job.started}")
print(f"Ended               : {query_job.ended}")

# Bytes processed
if query_job.total_bytes_processed is not None:
    print(
        f"Bytes processed     : "
        f"{query_job.total_bytes_processed:,} "
        f"({query_job.total_bytes_processed / 1024**3:.3f} GB)"
    )

if query_job.total_bytes_billed is not None:
    print(
        f"Bytes billed        : "
        f"{query_job.total_bytes_billed:,} "
        f"({query_job.total_bytes_billed / 1024**3:.3f} GB)"
    )


# ---------------------------------------------------------
# 4. Get newly created table
# ---------------------------------------------------------
table_id = (
    "prj-prod-dataplatform."
    "dap_ds_poweruser_playground."
    "beta_demo_model_cash_v1"
)

table = client.get_table(table_id)


# ---------------------------------------------------------
# 5. Table shape and metadata
# ---------------------------------------------------------
rows = table.num_rows
columns = len(table.schema)

print("\n========== TABLE DETAILS ==========")

print(f"Table               : {table_id}")
print(f"Shape               : ({rows:,}, {columns})")
print(f"Rows                : {rows:,}")
print(f"Columns             : {columns:,}")

print(
    f"Table size          : "
    f"{table.num_bytes:,} bytes "
    f"({table.num_bytes / 1024**2:.2f} MB)"
)

print(f"Created             : {table.created}")
print(f"Last modified       : {table.modified}")


# ---------------------------------------------------------
# 6. Print schema / column details
# ---------------------------------------------------------
print("\n========== SCHEMA ==========")

for i, field in enumerate(table.schema, start=1):
    print(
        f"{i:02d}. "
        f"{field.name:<30} "
        f"{field.field_type:<15} "
        f"{field.mode}"
    )

Starting BigQuery job...
BigQuery job completed successfully!

========== JOB DETAILS ==========
Job ID              : 7f3d22fc-e121-42f4-8322-816c8117354a
State               : DONE
Created             : 2026-10-08 03:43:45.788000+00:00
Started             : 2026-10-08 03:43:46.050000+00:00
Ended               : 2026-10-08 03:43:51.483000+00:00
Bytes processed     : 10,798,725,338 (10.057 GB)
Bytes billed        : 10,799,284,224 (10.058 GB)

========== TABLE DETAILS ==========
Table               : prj-prod-dataplatform.dap_ds_poweruser_playground.beta_demo_model_cash_v1
Shape               : (1,239,666, 26)
Rows                : 1,239,666
Columns             : 26
Table size          : 326,067,007 bytes (310.96 MB)
Created             : 2026-10-08 03:43:51.355000+00:00
Last modified       : 2026-10-08 03:43:51.355000+00:00

========== SCHEMA ==========
01. customerId                     STRING          NULLABLE
02. digitalLoanAccountId           STRING          NULLABLE
03. modelDispl